In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from medmnist import ChestMNIST
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, f1_score
import os

In [2]:
import urllib.request
import tarfile

os.makedirs('openi', exist_ok=True)

print("Téléchargement des rapports...")
urllib.request.urlretrieve(
    "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz",
    "openi/NLMCXR_reports.tgz"
)

with tarfile.open("openi/NLMCXR_reports.tgz", "r:gz") as tar:
    tar.extractall("openi/")

print("Rapports téléchargés ")
print(f"Nombre de rapports : {len(os.listdir('openi/ecgen-radiology'))}")

Téléchargement des rapports...
Rapports téléchargés 
Nombre de rapports : 3955


In [3]:
import xml.etree.ElementTree as ET
import random

def parse_report(filepath):
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        findings = root.find('.//AbstractText[@Label="FINDINGS"]')
        impression = root.find('.//AbstractText[@Label="IMPRESSION"]')
        
        text = ""
        if findings is not None and findings.text:
            text += findings.text + " "
        if impression is not None and impression.text:
            text += impression.text
        
        return text.strip() if text.strip() else "no finding"
    except:
        return "no finding"

# Charger tous les rapports
report_dir = "openi/ecgen-radiology"
report_files = [f for f in os.listdir(report_dir) if f.endswith('.xml')]

reports = []
for f in report_files:
    text = parse_report(os.path.join(report_dir, f))
    reports.append(text)

print(f"Nombre de rapports : {len(reports)}")
print(f"\nExemple de rapport :\n{reports[0][:300]}")

Nombre de rapports : 3955

Exemple de rapport :
The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no XXXX of a pleural effusion. There is no evidence of pneumothorax. Normal chest x-XXXX.


In [4]:
from torch.utils.data import Subset

torch.manual_seed(42)
np.random.seed(42)

transform_train = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

transform_test = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

train_dataset = ChestMNIST(split='train', download=True, size=64, transform=transform_train)
val_dataset   = ChestMNIST(split='val',   download=True, size=64, transform=transform_test)
test_dataset  = ChestMNIST(split='test',  download=True, size=64, transform=transform_test)

indices_10k = torch.load('models/indices_10k.pth')
train_dataset = Subset(train_dataset, indices_10k)
val_dataset   = Subset(val_dataset,   range(1000))
test_dataset  = Subset(test_dataset,  range(2000))

# Vocabulaire
all_words = set()
for r in reports:
    all_words.update(r.lower().split())
vocab = {w: i+1 for i, w in enumerate(sorted(all_words))}
vocab_size = len(vocab) + 1
print(f"Taille du vocabulaire : {vocab_size}")

def text_to_tensor(text, max_len=50):
    words = text.lower().split()[:max_len]
    ids = [vocab.get(w, 0) for w in words]
    ids += [0] * (max_len - len(ids))
    return torch.tensor(ids, dtype=torch.long)

class MultimodalDataset(Dataset):
    def __init__(self, img_dataset, reports):
        self.img_dataset = img_dataset
        self.reports = reports
    def __len__(self):
        return len(self.img_dataset)
    def __getitem__(self, idx):
        img, label = self.img_dataset[idx]
        report_idx = idx % len(self.reports)
        text = text_to_tensor(self.reports[report_idx])
        return img, text, torch.tensor(label, dtype=torch.float32)

train_mm = MultimodalDataset(train_dataset, reports)
val_mm   = MultimodalDataset(val_dataset, reports)
test_mm  = MultimodalDataset(test_dataset, reports)

train_loader = DataLoader(train_mm, batch_size=32, shuffle=True,
                          generator=torch.Generator().manual_seed(42))
val_loader   = DataLoader(val_mm,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_mm,  batch_size=32, shuffle=False)

print("Dataset multimodal prêt ")

Taille du vocabulaire : 3184
Dataset multimodal prêt 


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Modèle 1 : Image seule ──────────────────────────────
class ImageOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128*8*8, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 14)
        )
    def forward(self, img, text):
        return self.cnn(img)

# ── Modèle 2 : Texte seul ───────────────────────────────
class TextOnly(nn.Module):
    def __init__(self, vocab_size, embed_dim=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 14)
        )
    def forward(self, img, text):
        x = self.embed(text).mean(dim=1)
        return self.fc(x)

# ── Modèle 3 : Multimodal fusionné ──────────────────────
class MultimodalFusion(nn.Module):
    def __init__(self, vocab_size, embed_dim=64):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128*8*8, 256), nn.ReLU()
        )
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.text_fc = nn.Sequential(nn.Linear(embed_dim, 128), nn.ReLU())
        
        # Fusion tardive : concat image + texte
        self.classifier = nn.Sequential(
            nn.Linear(256 + 128, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 14)
        )
    
    def forward(self, img, text):
        img_feat = self.cnn(img)
        txt_feat = self.embed(text).mean(dim=1)
        txt_feat = self.text_fc(txt_feat)
        combined = torch.cat([img_feat, txt_feat], dim=1)
        return self.classifier(combined)

print("Modèles définis ")

Device : cpu
Modèles définis 


In [6]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, inputs, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        return (self.alpha * (1 - pt) ** self.gamma * bce_loss).mean()

def train_model(model, train_loader, val_loader, epochs=6, model_name='model'):
    criterion = FocalLoss(gamma=2.0, alpha=0.25)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    patience = 3
    counter = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for imgs, texts, labels in train_loader:
            imgs, texts, labels = imgs.to(device), texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs, texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, texts, labels in val_loader:
                imgs, texts, labels = imgs.to(device), texts.to(device), labels.to(device)
                outputs = model(imgs, texts)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        val_loss = val_loss / len(val_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'models/{model_name}_best.pth')
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping à l'epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(f'models/{model_name}_best.pth', map_location=device))
    print(f"Meilleur modèle {model_name} rechargé ")
    return train_losses, val_losses

print("=== Modèle 1 : Image seule ===")
model_img = ImageOnly().to(device)
losses_img = train_model(model_img, train_loader, val_loader, model_name='model_image')

print("\n=== Modèle 2 : Texte seul ===")
model_txt = TextOnly(vocab_size).to(device)
losses_txt = train_model(model_txt, train_loader, val_loader, model_name='model_text')

print("\n=== Modèle 3 : Multimodal ===")
model_mm = MultimodalFusion(vocab_size).to(device)
losses_mm = train_model(model_mm, train_loader, val_loader, model_name='model_multimodal')

=== Modèle 1 : Image seule ===
Epoch [1/6] | Train Loss: 0.0137 | Val Loss: 0.0120
Epoch [2/6] | Train Loss: 0.0127 | Val Loss: 0.0118
Epoch [3/6] | Train Loss: 0.0126 | Val Loss: 0.0120
Epoch [4/6] | Train Loss: 0.0126 | Val Loss: 0.0119
Epoch [5/6] | Train Loss: 0.0125 | Val Loss: 0.0117
Epoch [6/6] | Train Loss: 0.0125 | Val Loss: 0.0118
Meilleur modèle model_image rechargé 

=== Modèle 2 : Texte seul ===
Epoch [1/6] | Train Loss: 0.0158 | Val Loss: 0.0127
Epoch [2/6] | Train Loss: 0.0131 | Val Loss: 0.0124
Epoch [3/6] | Train Loss: 0.0129 | Val Loss: 0.0124
Epoch [4/6] | Train Loss: 0.0129 | Val Loss: 0.0124
Epoch [5/6] | Train Loss: 0.0128 | Val Loss: 0.0124
Epoch [6/6] | Train Loss: 0.0128 | Val Loss: 0.0124
Early stopping à l'epoch 6
Meilleur modèle model_text rechargé 

=== Modèle 3 : Multimodal ===
Epoch [1/6] | Train Loss: 0.0141 | Val Loss: 0.0121
Epoch [2/6] | Train Loss: 0.0129 | Val Loss: 0.0120
Epoch [3/6] | Train Loss: 0.0127 | Val Loss: 0.0119
Epoch [4/6] | Train Loss:

In [7]:
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score

label_names = ['atelectasis', 'cardiomegaly', 'effusion', 'infiltration',
               'mass', 'nodule', 'pneumonia', 'pneumothorax',
               'consolidation', 'edema', 'emphysema', 'fibrosis', 'pleural', 'hernia']

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, texts, labels in loader:
            imgs, texts = imgs.to(device), texts.to(device)
            outputs = torch.sigmoid(model(imgs, texts))
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_preds_binary = (all_preds >= 0.1).astype(int)
    aucs = [roc_auc_score(all_labels[:, i], all_preds[:, i]) for i in range(14)]
    f1 = f1_score(all_labels, all_preds_binary, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds_binary, average='macro', zero_division=0)
    precision = precision_score(all_labels, all_preds_binary, average='macro', zero_division=0)
    return np.mean(aucs), f1, recall, precision

auc_img, f1_img, rec_img, prec_img = evaluate(model_img, test_loader)
auc_txt, f1_txt, rec_txt, prec_txt = evaluate(model_txt, test_loader)
auc_mm,  f1_mm,  rec_mm,  prec_mm  = evaluate(model_mm,  test_loader)

print(f"{'Modèle':<20} | {'AUC':>6} | {'F1':>6} | {'Recall':>6} | {'Precision':>9}")
print("=" * 60)
print(f"{'Image seule':<20} | {auc_img:>6.4f} | {f1_img:>6.4f} | {rec_img:>6.4f} | {prec_img:>9.4f}")
print(f"{'Texte seul':<20} | {auc_txt:>6.4f} | {f1_txt:>6.4f} | {rec_txt:>6.4f} | {prec_txt:>9.4f}")
print(f"{'Multimodal':<20} | {auc_mm:>6.4f}  | {f1_mm:>6.4f} | {rec_mm:>6.4f} | {prec_mm:>9.4f}")

Modèle               |    AUC |     F1 | Recall | Precision
Image seule          | 0.6379 | 0.0990 | 0.9448 |    0.0543
Texte seul           | 0.4885 | 0.0973 | 0.9405 |    0.0534
Multimodal           | 0.6239  | 0.0972 | 0.9085 |    0.0533


In [8]:
torch.save(model_img.state_dict(), 'models/model_image.pth')
torch.save(model_txt.state_dict(), 'models/model_text.pth')
torch.save(model_mm.state_dict(), 'models/model_multimodal.pth')
print("Modèles sauvegardés ")

Modèles sauvegardés 


In [11]:
import pickle
with open('models/vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print("Vocabulaire sauvegardé ")

Vocabulaire sauvegardé 
